# Chapter 8: Transfer Learning

This notebook accompanies **Chapter 8** of the lecture notes.

> Last lecture you trained an encoder from scratch and looked at what the bottleneck had organised. Today the encoder arrives pretrained — someone else paid for it — and the lecture is about what to do with it. Freeze it, probe it, fine-tune it. Bridge it to a second modality. Compose it with pieces it was never trained alongside. The notebook walks the same arc as the published recipe behind CLIP, SAM, and the rest — only the model and dataset shrink so everything fits on your laptop.

**Agenda**

🧱 · 🪜 · 🎯 · 🔗 · 🧩 · 🏁

**Take it from here:** 🔬 · 🧪 · 🦙

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones. Nothing here downloads weights — we pretrain a tiny backbone in seconds inside this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '../..')
from plot_style import *
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from scipy.optimize import minimize
from checks import (
    check_encode, check_linear_probe, check_attribute_table,
    check_class_centroids, check_attribute_bridge, check_cosine_classify,
    check_contrastive_loss, check_topk_retrieve,
)

RNG = np.random.default_rng(0)

## 🧱 Backbones

A backbone is an encoder trained on one task, then reused as a feature extractor for another. A small head on top — usually one linear layer — supplies the predictions for the new task. The backbone supplies the *representation*; the head supplies the *decision*.

The reason this works at all is the same point we made about embeddings in chapter 6. A bottlenecked encoder cannot memorise individual inputs; it has to discover regularities that hold across the training distribution. Those regularities are exactly what a downstream model needs to perform well on new, unseen data.

### A short origin story

Before backbones, vision systems were stacks of *hand-crafted features* — SIFT, HOG, Haar wavelets — designed by hand and pasted into pipelines per task. The first turning point came in 2014: two papers (Razavian et al., Yosinski et al.) showed that *off-the-shelf* features from a network pretrained on ImageNet beat the best hand-crafted features almost everywhere they were measured. ImageNet pretraining became the default starting point for vision.

The second turning point was **scale**. Once the encoders were trained on web-scale data instead of ImageNet's curated 1.4M, they stopped looking like task-specific tools and started looking like general-purpose substrates that downstream tasks plugged into. By that point the community had a new name for them — *foundation models* — and a new vocabulary (zero-shot, prompting, in-context learning) for how you actually use them. **Foundation models are backbones with three additions: scale, breadth, and (often) cross-modal alignment. No new architecture.**

| Backbone | Year | Params | Pretraining data | Transfer style |
|---|---|---|---|---|
| AlexNet / VGG / ResNet-50 | 2012–2015 | 60M / 138M / 25M | ImageNet, 1.4M images, 1000 classes | fine-tune the head |
| BERT-base | 2018 | 110M | Books + Wikipedia, ~3.3B tokens | fine-tune the whole model per task |
| GPT-3 | 2020 | 175B | ~500B tokens, web + books + code | in-context, no gradient updates |
| CLIP ViT-L/14 | 2021 | ~430M (vision + text) | 400M image-caption pairs from the web | zero-shot via text prompts |
| DINOv2 ViT-g | 2023 | ~1.1B | 142M curated images, no labels | linear probe on frozen features |
| SAM ViT-H | 2023 | 636M | 11M images, 1B masks | promptable segmentation, zero-shot |
| Llama-3-70B | 2024 | 70B | ~15T tokens | LoRA / QLoRA / in-context |

These models do *exactly* what we are about to do here, only at a six-orders-of-magnitude-larger scale. The mechanic on this page is the published recipe.

### A toy backbone we can train in seconds

We pretrain on the eight digits 2 through 9 — that is the "ImageNet" of this notebook. The two unseen classes 0 and 1 are held out completely; the backbone never sees a 0 or a 1 during pretraining. The whole rest of the notebook is about reaching those held-out classes through what the backbone learned on the others.

In [ ]:
digits = load_digits()
X_all, y_all = digits.data / 16.0, digits.target  # X is (1797, 64), values in [0, 1]

# Source classes (used to pretrain the backbone): digits 2..9
src_mask = (y_all >= 2)
X_src, y_src = X_all[src_mask], y_all[src_mask]

# Target classes (held out from pretraining): digits 0 and 1
tgt_mask = (y_all <= 1)
X_tgt, y_tgt = X_all[tgt_mask], y_all[tgt_mask]

print(f'Source pool : {X_src.shape}, classes {sorted(set(y_src))}')
print(f'Target pool : {X_tgt.shape}, classes {sorted(set(y_tgt))}')

In [ ]:
# Pretrain the backbone — supervised classification on the source classes.
# Hidden sizes (32, 16) means the last hidden layer is 16-dimensional;
# that 16-d activation is the embedding we will reuse downstream.
backbone = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=80,
    random_state=0,
)
backbone.fit(X_src, y_src)
print(f'Pretraining accuracy on source classes: {backbone.score(X_src, y_src):.3f}')
print(f'Layer shapes (W, b):')
for i, (W, b) in enumerate(zip(backbone.coefs_, backbone.intercepts_)):
    print(f'  layer {i}: W {W.shape},  b {b.shape}')

### Strip the head and embed

The backbone has three weight matrices: 64 → 32, 32 → 16, 16 → 8. The first two are the *encoder*; the last one is the *classification head*. Transfer starts by throwing the head away and treating the 16-d activation as a general-purpose representation.

> The head was trained to separate the eight source classes. Why would the activations one layer earlier — never directly supervised — also be useful for separating digits the model has never seen?

<details><summary>Thought</summary>

Because the only way the head can do its job is if the layer below it has already organised the inputs into something close to linearly separable. A representation good enough to separate eight digits has to carry features like loops, vertical strokes, and curvature — features that are not specific to those eight classes. A 0 (round, one loop) and a 1 (vertical stroke, no loop) are described in exactly the same vocabulary the backbone already learned for 6, 8, 9, 4, and 7.
</details>

Implement `encode`. Read the trained backbone's weights, run a forward pass through the first two layers (the encoder), apply the activation, and L2-normalise each row. The output should be `(n, 16)` with every row on the unit sphere. Useful operations: `mlp.coefs_`, `mlp.intercepts_`, `np.maximum(0, z)` for ReLU, `np.linalg.norm(..., axis=1, keepdims=True)`.

In [ ]:
def encode(X, mlp):
    """Forward-pass X through the encoder layers of mlp; return L2-normalised embeddings."""
    W1, b1 = mlp.coefs_[0], mlp.intercepts_[0]
    W2, b2 = mlp.coefs_[1], mlp.intercepts_[1]
    h1 = np.maximum(0, X @ W1 + b1)
    Z  = np.maximum(0, h1 @ W2 + b2)
    return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)


check_encode(encode, X_src[:50], backbone)

In [ ]:
_Z = encode(X_src, backbone)
if _Z is None:
    print('⬜ Implement encode above first.')
else:
    # PCA project to 2D for plotting (use numpy SVD; sklearn PCA would also work).
    _Zc = _Z - _Z.mean(axis=0)
    _U, _S, _Vt = np.linalg.svd(_Zc, full_matrices=False)
    _Z2 = _Zc @ _Vt[:2].T

    fig, ax = plt.subplots(figsize=(7, 6))
    _cmap = plt.get_cmap('tab10', 10)
    for d in sorted(set(y_src)):
        m = y_src == d
        ax.scatter(_Z2[m, 0], _Z2[m, 1], color=_cmap(d), s=10, linewidths=0,
                   alpha=0.7, label=str(d))
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
    ax.set_title('Backbone embeddings of source classes (2..9), PCA-projected', fontsize=10, color=_GOLDEN)
    ax.legend(frameon=False, ncol=4, fontsize=8, labelcolor=_TEXT)
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()

**Observe:**
- The eight source digits separate into eight clusters in the 16-d space, even though we are looking at a 2-d shadow of it. The embedding reflects digit identity.
- Some neighbours make visual sense — 6 sits near 8 and 9 because all three are loop-heavy; 4 and 7 sit near 1's natural position because all three are stroke-dominated.
- That semantic geometry is the whole asset. Everything in the rest of the notebook is a different way of cashing it in for a downstream task.

**At scale.** Run the same forward-pass-and-L2-normalise on a frozen ResNet-50 (25M params, ImageNet pretrained) and the scatter looks similar — clusters per ImageNet class, and visually similar classes sitting next to each other (huskies near wolves, cellos near violins). Replace the backbone with DINOv2 (1.1B params, no labels, 142M images) and the clusters become semantic across far more categories than the model was ever asked to label. The PCA scatter is the same diagnostic; the substrate is what changed.

## 🪜 The Transfer Continuum

The lecture lists the continuum: frozen feature extraction, linear probing, partial fine-tuning, full fine-tuning, PEFT (LoRA / adapters / BitFit), prompting, in-context learning. It is one axis, ordered by *how much of the model you are willing to update* and *how much labelled target data you have*. The two extremes are the most informative for a small notebook: a frozen backbone with a tiny linear classifier on top, versus an MLP trained from scratch on the same target data with no transfer at all.

The recipe-box prediction from the lecture is concrete: when target labels are scarce, the frozen probe wins because the backbone is doing most of the work; when target labels are plentiful, training from scratch eventually catches up because the model can learn task-specific features the backbone never had.

In [ ]:
# Build a target train/test split of digits 0 and 1.
# Test set is fixed; train set is sampled at different K (shots per class).
_perm = RNG.permutation(len(X_tgt))
X_tgt_p, y_tgt_p = X_tgt[_perm], y_tgt[_perm]

# Hold out 60 examples per class as the test set.
_idx_test_0 = np.where(y_tgt_p == 0)[0][:60]
_idx_test_1 = np.where(y_tgt_p == 1)[0][:60]
test_idx = np.concatenate([_idx_test_0, _idx_test_1])
pool_idx = np.array([i for i in range(len(X_tgt_p)) if i not in set(test_idx)])

X_tgt_test, y_tgt_test = X_tgt_p[test_idx], y_tgt_p[test_idx]
X_tgt_pool, y_tgt_pool = X_tgt_p[pool_idx], y_tgt_p[pool_idx]
print(f'Target test : {X_tgt_test.shape}, balanced over 0/1')
print(f'Target pool : {X_tgt_pool.shape}, available shots per class')

### Linear probe on frozen embeddings

The simplest non-trivial transfer strategy: freeze the backbone, embed the target training set, train a logistic regression on top.

> The probe is a single linear layer on top of a 16-d frozen embedding. With K = 5 examples per class, what limits the probe's accuracy — the size of K, or the quality of the embedding it sits on?

<details><summary>Thought</summary>

The embedding. With K = 5 the probe has barely any data to play with, but it is also a very small model — sixteen weights and a bias per class. The probe will fit in milliseconds; what determines whether it can separate the classes is whether the 16-d embedding already places 0s and 1s on different sides of a hyperplane. If the backbone learned features like "has loop" and "vertical stroke", a linear boundary separates them trivially. If it didn't, no amount of K helps.
</details>

Implement `linear_probe`. Fit a logistic regression on the embedded training set, score it on the embedded test set, and return the accuracy as a float. Useful: `LogisticRegression(max_iter=2000).fit(...).score(...)`.

In [ ]:
def linear_probe(emb_train, y_train, emb_test, y_test):
    """Fit logistic regression on (emb_train, y_train), return test accuracy."""
    clf = LogisticRegression(max_iter=2000)
    clf.fit(emb_train, y_train)
    return float(clf.score(emb_test, y_test))


_rng = np.random.default_rng(1)
def _k_shot(K, rng=_rng):
    idx = []
    for c in (0, 1):
        ci = np.where(y_tgt_pool == c)[0]
        idx.extend(rng.choice(ci, size=K, replace=False))
    idx = np.array(idx)
    return X_tgt_pool[idx], y_tgt_pool[idx]

_Xs, _ys = _k_shot(5)
_emb_train = encode(_Xs, backbone)
_emb_test  = encode(X_tgt_test, backbone)
check_linear_probe(linear_probe, _emb_train, _ys, _emb_test, y_tgt_test)

In [ ]:
# K-shot sweep: frozen+probe vs train-from-scratch on raw pixels.
# Both classifiers see the same (X_train, y_train); the difference is whether
# they sit on top of the backbone's embedding or on the raw 64-d pixel input.
_Ks = [1, 2, 5, 10, 20, 50]
_n_repeats = 5
_acc_probe   = np.zeros((len(_Ks), _n_repeats))
_acc_scratch = np.zeros((len(_Ks), _n_repeats))

if linear_probe(_emb_train, _ys, _emb_test, y_tgt_test) is None:
    print('⬜ Implement linear_probe above first.')
else:
    for ki, K in enumerate(_Ks):
        for r in range(_n_repeats):
            rng = np.random.default_rng(100 + r)
            Xs, ys = _k_shot(K, rng=rng)
            # Frozen backbone + logistic-regression probe.
            _acc_probe[ki, r] = linear_probe(
                encode(Xs, backbone), ys, _emb_test, y_tgt_test
            )
            # Train-from-scratch baseline: same architecture as the backbone,
            # no transfer, fit on the K-shot raw-pixel data.
            _scratch = MLPClassifier(
                hidden_layer_sizes=(32, 16), activation='relu',
                solver='adam', learning_rate_init=1e-3,
                max_iter=300, random_state=r,
            ).fit(Xs, ys)
            _acc_scratch[ki, r] = _scratch.score(X_tgt_test, y_tgt_test)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.errorbar(_Ks, _acc_probe.mean(axis=1),   yerr=_acc_probe.std(axis=1),
                color=_ACCENT, marker='o', linewidth=1.4, capsize=3,
                label='frozen backbone + probe')
    ax.errorbar(_Ks, _acc_scratch.mean(axis=1), yerr=_acc_scratch.std(axis=1),
                color=_TERRA,  marker='s', linewidth=1.4, capsize=3,
                label='train MLP from scratch on raw pixels')
    ax.set_xscale('log'); ax.set_xlabel('shots per class (K)')
    ax.set_ylabel('test accuracy on 0 vs 1')
    ax.set_title('K-shot transfer: frozen backbone vs from-scratch', fontsize=10, color=_GOLDEN)
    ax.set_ylim(0.4, 1.02)
    ax.legend(frameon=False, labelcolor=_TEXT)
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()

**Observe:**
- At K = 1 the probe is already in the high 90s. The backbone's embedding has done so much of the work that one labelled example per class is enough for a linear boundary.
- The from-scratch baseline starts near chance and only catches up once K is large. With ten or twenty shots it has nothing to learn from.
- This is the recipe box from the lecture in one picture. Small target dataset, similar domain → freeze the backbone. Large target dataset, different domain → fine-tuning becomes worth the cost. Everything in between is a continuum, and the choice is a hyperparameter of how much you trust the backbone's features.

**At scale: PEFT.** When the backbone is a 70B-parameter LLM, "fine-tune the whole thing" means rewriting hundreds of gigabytes of weights for every new task. That is infeasible, so the field invented *parameter-efficient fine-tuning* — freeze the model, learn a small additive update. **LoRA** decomposes each weight matrix's update as the product of two skinny matrices `BA` (typically rank 8–64), training fewer than 1% of the original parameters. **Adapters** insert tiny bottleneck modules between transformer layers. **BitFit** updates only the biases. Same continuum as our K-shot plot — the difference is that on a 70B model, even "linear probe" is no longer trivial, and PEFT becomes the practical default. **Prompting and in-context learning** sit further along the same axis: zero parameter updates, behaviour shaped entirely by the input. We name them here and meet ICL again in the LLM lectures.

## 🎯 Lampert: Attribute-Based Zero-Shot

The probe still needed a few labelled examples of 0 and 1. *Zero-shot* asks for none. Lampert's 2009 idea was to bridge images to unseen classes through a hand-built attribute table: every class — seen or unseen — is described by the same fixed vocabulary of attributes, and a classifier learned on seen classes carries over because the bridge is shared.

We use the same recipe. The attributes are five binary features that distinguish digits visually: presence of a closed loop, presence of two loops, a strong vertical stroke, a strong horizontal stroke, and overall curviness. The source rows (digits 2 through 9) are already filled in below. Your job is the two target rows, 0 and 1, which is exactly what a Lampert-era researcher would have done by hand to define a new class.

> Why does a class description in the same attribute vocabulary as the seen classes give a working classifier for an unseen class, even though the model has never seen one?

<details><summary>Thought</summary>

Because the bridge between attributes and embeddings was learned on the seen classes, and that bridge is what gets reused. If the source rows tell us that "has loop" maps consistently to a particular direction in embedding space, and "vertical stroke" to a different direction, then any new class — described in those same coordinates — gets a predicted location in embedding space without ever being seen. Classification is then nearest-neighbour against that prediction.
</details>

In [ ]:
# Attribute names — the shared vocabulary.
attr_names = ['has_loop', 'two_loops', 'vertical_stroke', 'horizontal_stroke', 'curvy']

# Source-class attributes: digits 2..9. These are pre-filled by the instructor;
# students are not expected to argue with them in this exercise.
attrs_source = {
    2: [0, 0, 0, 1, 1],   # curvy with a horizontal bar at the bottom
    3: [0, 0, 0, 0, 1],   # all curves, no loops, no straight strokes
    4: [0, 0, 1, 1, 0],   # vertical and horizontal strokes
    5: [0, 0, 0, 1, 1],   # curve plus a horizontal stroke at the top
    6: [1, 0, 0, 0, 1],   # one loop at the bottom, curvy
    7: [0, 0, 1, 1, 0],   # diagonal/vertical stroke and a horizontal top
    8: [1, 1, 0, 0, 1],   # two loops, fully curvy
    9: [1, 0, 1, 0, 1],   # one loop on top with a tail going down
}

# Confirm the source table is well-formed.
for d, row in attrs_source.items():
    assert len(row) == len(attr_names)
print('Source attribute table:')
for d in sorted(attrs_source):
    print(f'  {d}: ' + '  '.join(f'{n}={v}' for n, v in zip(attr_names, attrs_source[d])))

### Fill in the unseen-class rows

Define `attrs_target` as a dict mapping each held-out digit (`0` and `1`) to a five-element list of 0s and 1s, in the same order as `attr_names`. Think of how each digit looks: a 0 is a single closed loop with no straight strokes; a 1 is a single vertical stroke with no loops.

The checker validates the format and the rough shape of each row, not the *exact* answer — small disagreements are fine.

In [ ]:
attrs_target = {
    # 0 is round, has one closed loop, no straight strokes, fully curvy.
    0: [1, 0, 0, 0, 1],
    # 1 is dominated by a vertical stroke, no loops, no curves, no horizontal bar.
    1: [0, 0, 1, 0, 0],
}


check_attribute_table(attrs_target, attr_names)

### Class centroids in embedding space

The bridge needs anchor points. For each source class, compute the mean of its embeddings — the *prototype* of that class. The attribute-to-embedding map will be fitted against these prototypes.

In [ ]:
def class_centroids(emb, labels):
    """Return a dict {class_label: centroid_vector} of L2-normalised mean embeddings."""
    out = {}
    for label in sorted(set(labels.tolist() if hasattr(labels, 'tolist') else labels)):
        mask = (np.asarray(labels) == label)
        c = emb[mask].mean(axis=0)
        out[int(label)] = c / (np.linalg.norm(c) + 1e-12)
    return out


_emb_src = encode(X_src, backbone)
check_class_centroids(class_centroids, _emb_src, y_src)

### Attribute-to-embedding bridge

Fit a linear map W from attribute vectors to embedding centroids using the source classes. Closed form: stack the source attribute rows into a matrix A (shape `n_src_classes × n_attrs`) and the corresponding centroids into a matrix C (shape `n_src_classes × 16`). Solve `A @ W = C` for W in the least-squares sense. Then apply W to the unseen-class attribute rows to predict their centroids.

> The eight source rows give us eight equations in five unknowns per output dimension. Why does that work, and what would happen if we had only three source classes?

<details><summary>Thought</summary>

Eight equations in five unknowns is overdetermined per output dimension, so least-squares finds the W that best fits all of them simultaneously — generalisation by averaging. Three source classes would leave the system underdetermined (three equations, five unknowns), so W could fit the source perfectly while behaving arbitrarily on unseen attribute combinations. Source breadth is the silent ingredient that makes attribute-based zero-shot work.
</details>

In [ ]:
def attribute_bridge(attrs_source, centroids_source, attrs_target):
    """Fit W via least squares on source pairs, apply to target attribute rows."""
    src_keys = sorted(attrs_source.keys())
    A = np.array([attrs_source[k] for k in src_keys], dtype=np.float64)
    C = np.array([np.asarray(centroids_source[k]) for k in src_keys], dtype=np.float64)
    W, *_ = np.linalg.lstsq(A, C, rcond=None)
    out = {}
    for label, attr_row in attrs_target.items():
        pred = np.asarray(attr_row, dtype=np.float64) @ W
        out[label] = pred / (np.linalg.norm(pred) + 1e-12)
    return out


_centroids_src = class_centroids(_emb_src, y_src)
if _centroids_src is not None and attrs_target.get(0) is not None and attrs_target.get(1) is not None:
    check_attribute_bridge(attribute_bridge, attrs_source, _centroids_src, attrs_target)
else:
    print('⬜ Need encode, attrs_target, and class_centroids implemented first.')

### Cosine classify and read the score

With predicted centroids for 0 and 1 in hand, classify each test image by which centroid its embedding is closest to under cosine similarity. Both embeddings and centroids are unit-norm, so cosine similarity is just a dot product.

In [ ]:
def cosine_classify(query_emb, class_emb_dict):
    """Return predicted labels by argmax cosine similarity."""
    labels = sorted(class_emb_dict.keys())
    M = np.stack([np.asarray(class_emb_dict[l]) for l in labels])
    S = query_emb @ M.T
    return np.array([labels[i] for i in np.argmax(S, axis=1)])


_q = np.array([[1.0, 0.0], [0.0, 1.0]])
_c = {'a': np.array([1.0, 0.0]), 'b': np.array([0.0, 1.0])}
check_cosine_classify(cosine_classify, _q, _c, expected=np.array(['a', 'b']))

In [ ]:
# Run zero-shot on the held-out target set.
_emb_tgt_test = encode(X_tgt_test, backbone)
_centroids_src = class_centroids(_emb_src, y_src)
_predicted_tgt = attribute_bridge(attrs_source, _centroids_src, attrs_target) if _centroids_src is not None else None

if _predicted_tgt is None:
    print('⬜ Need attribute_bridge implemented first.')
else:
    _preds = cosine_classify(_emb_tgt_test, _predicted_tgt)
    if _preds is None:
        print('⬜ Need cosine_classify implemented first.')
    else:
        _acc = float((_preds == y_tgt_test).mean())
        print(f'Lampert zero-shot accuracy on (0, 1): {_acc:.3f}')

        # Confusion matrix.
        _cm = np.zeros((2, 2), dtype=int)
        for t, p in zip(y_tgt_test, _preds):
            _cm[int(t), int(p)] += 1
        print(f'Confusion matrix [rows=true, cols=pred]:')
        print(f'         pred=0  pred=1')
        print(f' true=0  {_cm[0,0]:>6}  {_cm[0,1]:>6}')
        print(f' true=1  {_cm[1,0]:>6}  {_cm[1,1]:>6}')

**Observe:**
- The model classifies digits it was never trained on, using only an attribute description as the bridge. No labelled 0s, no labelled 1s.
- The accuracy depends on whether the source attribute table actually captured the directions the backbone uses internally. If "has loop" is consistently encoded as a clear axis in the embedding, generalisation is clean. If two source classes share an attribute pattern that doesn't actually look the same in the embedding, the bridge becomes noisy.
- Lampert is the historical answer to "how do you reach unseen classes". The next section asks the model to learn the bridge for itself.

**At scale.** Lampert's original 2009 paper bridged 50 animal classes through 85 attributes — `has_horns`, `lives_in_water`, `is_brown` — hand-curated by the authors. That handcrafting cost is exactly what makes the approach unscalable: every new domain needs a new attribute vocabulary built by experts. Replacing the attribute table with **natural language**, and the least-squares fit with **contrastive training on web-scale image-text pairs**, is the move that gets you CLIP. Same picture, different bridge.

## 🔗 Mini-CLIP: Contrastive Cross-Modal Alignment

CLIP replaces the hand-built attribute bridge with a learned one. Two encoders — one for images, one for text — are trained jointly so that an image and its matching caption land on the same point in a shared space, and unrelated pairs are pushed apart. The training signal is a single matrix: pair up N images with N captions, compute the N×N similarity matrix, and ask the model to make the diagonal big and everything else small.

We do that here with the same backbone embeddings (image side) and the same attribute vectors (taking the place of text). The image encoder is frozen — we already have the embeddings. The "text" encoder is a single linear layer mapping attribute vectors to the same 16-d space.

### The diagonal contrastive loss

The InfoNCE / CLIP loss takes a batch of N image embeddings and N text embeddings (paired by index), forms an N×N similarity matrix scaled by a temperature, and applies cross-entropy in both directions — image-to-text and text-to-image — encouraging the diagonal to dominate each row and each column.

Implement `contrastive_loss(img_embs, attr_embs, tau)`. Both inputs are `(N, d)` and assumed L2-normalised. Compute `S = (img_embs @ attr_embs.T) / tau`, treat each row as logits over text candidates with the correct answer being the diagonal index, compute cross-entropy, do the same column-wise, and return their average.

Useful operations: `np.log(np.exp(s).sum(axis=1))` is the log-sum-exp; subtract `max` first for numerical stability. The cross-entropy of row `i` is `-S[i, i] + logsumexp(S[i, :])`.

In [ ]:
def contrastive_loss(img_embs, attr_embs, tau=0.1):
    """Symmetric InfoNCE loss on (N, d) paired embeddings."""
    S = (img_embs @ attr_embs.T) / tau
    m_r = S.max(axis=1, keepdims=True)
    lse_r = np.log(np.exp(S - m_r).sum(axis=1)) + m_r.ravel()
    m_c = S.max(axis=0, keepdims=True)
    lse_c = np.log(np.exp(S - m_c).sum(axis=0)) + m_c.ravel()
    diag = np.diag(S)
    loss_i2t = (lse_r - diag).mean()
    loss_t2i = (lse_c - diag).mean()
    return float(0.5 * (loss_i2t + loss_t2i))


_rng = np.random.default_rng(0)
_v = _rng.normal(size=(8, 16))
_v = _v / np.linalg.norm(_v, axis=1, keepdims=True)
check_contrastive_loss(contrastive_loss, _v)

In [ ]:
# Train the attribute encoder W (shape (n_attrs, d_emb)) by minimising the
# contrastive loss on pairs (image_embedding, attribute_vector_of_its_class).
# Free parameters: 5 attributes * 16 dim = 80, small enough for L-BFGS-B.

# Build paired training set: every source image with the attribute row of its class.
_emb_src = encode(X_src, backbone)
_A_train = np.array([attrs_source[int(y)] for y in y_src], dtype=np.float64)
print(f'Pairs: img embs {_emb_src.shape},  attr vectors {_A_train.shape}')

# Subsample to keep optimisation fast (~300 pairs is plenty given 80 params).
_n_pairs = 300
_idx_pairs = RNG.choice(len(_emb_src), size=_n_pairs, replace=False)
_img_pairs  = _emb_src[_idx_pairs]
_attr_pairs = _A_train[_idx_pairs]

_n_attrs, _d_emb = len(attr_names), 16
def _wrapped_loss(w_flat):
    W = w_flat.reshape(_n_attrs, _d_emb)
    text_embs = _attr_pairs @ W
    text_embs = text_embs / (np.linalg.norm(text_embs, axis=1, keepdims=True) + 1e-8)
    return float(contrastive_loss(_img_pairs, text_embs, tau=0.1))

# Test the loss is callable before optimising.
_w0 = RNG.normal(size=_n_attrs * _d_emb) * 0.1
if contrastive_loss(_img_pairs, _attr_pairs @ _w0.reshape(_n_attrs, _d_emb), tau=0.1) is None:
    print('⬜ Implement contrastive_loss above first.')
    W_attr = None
else:
    print(f'Initial loss : {_wrapped_loss(_w0):.4f}')
    _result = minimize(_wrapped_loss, _w0, method='L-BFGS-B',
                       options={'maxiter': 60, 'disp': False})
    W_attr = _result.x.reshape(_n_attrs, _d_emb)
    print(f'Final loss   : {_wrapped_loss(_result.x):.4f}')
    print(f'Iterations   : {_result.nit}')

In [ ]:
# Visualise the similarity matrix S before and after training, on a small
# diagnostic batch with one image per source class.
if W_attr is None:
    print('⬜ Train W_attr above first.')
else:
    _diag_idx = []
    for c in sorted(set(y_src)):
        ci = np.where(y_src == c)[0][0]
        _diag_idx.append(ci)
    _diag_idx = np.array(_diag_idx)
    _img_diag  = _emb_src[_diag_idx]
    _attr_diag = _A_train[_diag_idx]

    def _sim_with(W):
        text = _attr_diag @ W
        text = text / (np.linalg.norm(text, axis=1, keepdims=True) + 1e-8)
        return _img_diag @ text.T

    _S_before = _sim_with(_w0.reshape(_n_attrs, _d_emb))
    _S_after  = _sim_with(W_attr)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    for ax, S, title in [(axes[0], _S_before, 'Before training'),
                         (axes[1], _S_after,  'After training')]:
        im = ax.imshow(S, cmap='magma', vmin=-1, vmax=1, aspect='equal')
        ax.set_xticks(range(8)); ax.set_yticks(range(8))
        ax.set_xticklabels(sorted(set(y_src))); ax.set_yticklabels(sorted(set(y_src)))
        ax.set_xlabel('attribute (class)'); ax.set_ylabel('image (class)')
        ax.set_title(title, fontsize=10, color=_GOLDEN)
        for s in ax.spines.values(): s.set_visible(False)
        ax.tick_params(length=0)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

In [ ]:
# Re-run zero-shot on (0, 1) using the LEARNED attribute encoder.
if W_attr is None:
    print('⬜ Train W_attr above first.')
elif attrs_target.get(0) is None or attrs_target.get(1) is None:
    print('⬜ Fill in attrs_target rows above first.')
else:
    _A_tgt = np.array([attrs_target[c] for c in (0, 1)], dtype=np.float64)
    _txt_tgt = _A_tgt @ W_attr
    _txt_tgt = _txt_tgt / (np.linalg.norm(_txt_tgt, axis=1, keepdims=True) + 1e-8)
    _learned_centroids = {0: _txt_tgt[0], 1: _txt_tgt[1]}

    _preds_clip = cosine_classify(_emb_tgt_test, _learned_centroids)
    if _preds_clip is None:
        print('⬜ Need cosine_classify implemented first.')
    else:
        _acc_clip = float((_preds_clip == y_tgt_test).mean())
        # Recompute Lampert baseline for direct comparison.
        _predicted_tgt = attribute_bridge(attrs_source, class_centroids(_emb_src, y_src), attrs_target)
        _acc_lampert = float((cosine_classify(_emb_tgt_test, _predicted_tgt) == y_tgt_test).mean())
        print(f'Lampert  (least-squares bridge)     : {_acc_lampert:.3f}')
        print(f'Mini-CLIP (contrastive bridge)      : {_acc_clip:.3f}')

**Observe:**
- The before-training similarity matrix is noise. The after-training matrix has the diagonal lit and the off-diagonal damped — the contrastive objective in one image.
- Both bridges use the same source data and the same attribute table; only the *fitting criterion* differs. Least-squares minimises a regression error; contrastive minimises a discrimination error. The discrimination error is closer to what we actually use the bridge for at test time.

**At scale.** The published version of this picture is below. The mechanics on this page — diagonal contrastive loss, paired encoders, L2-normalised embeddings, temperature-scaled cosine — are *exactly* the mechanics of the published models. The difference is the substrate.

| | Our mini-CLIP | CLIP ViT-L/14 (2021) | SigLIP / ALIGN | LLaVA / GPT-4V (multimodal LLMs) |
|---|---|---|---|---|
| Image encoder | 2-layer MLP, ~3K params | ViT-L, ~300M params | ViT-L/H, 300M–2B | ViT + LLM, billions |
| Text encoder | linear, 80 params | Transformer, ~120M | Transformer, hundreds of M | the LLM itself |
| Training pairs | 300 image-attribute | 400M image-caption | up to 1B image-caption | image + interleaved text |
| Embedding dim | 16 | 768 | up to 1152 | shared with the LLM |
| Loss | symmetric InfoNCE | symmetric InfoNCE | sigmoid (SigLIP) / InfoNCE | next-token + alignment |
| Training time | ~1 second on a CPU | weeks on hundreds of GPUs | similar | months on thousand-GPU clusters |

SigLIP and ALIGN are essentially "same recipe, different scaling and loss tweaks". The multimodal LLMs (LLaVA, GPT-4V, Gemini) drop the separate text encoder and reuse the LLM itself as the text side, with cross-modal alignment learned during instruction tuning — the same idea, taken further. We meet that line again in the LLM lectures.

## 🧩 Composing the Pieces

Three things were built above: a backbone (image encoder), an attribute encoder (the "text" side of mini-CLIP), and a similarity-based classifier. None of them was trained for the task we are about to give them — *retrieve every digit in the dataset that matches a free-form attribute query, then describe what you found*.

We compose: the attribute query goes through the trained text encoder; cosine ranks every image in the dataset; the top matches are clustered with k-means (from the lecture-02 notebook); cluster centroids reveal what the retrieved set has in common. None of those steps required new training.

### Top-K retrieval

Implement `topk_retrieve(query_emb, all_embs, k)`. Return the indices of the k rows of `all_embs` most similar to `query_emb` under cosine similarity, in descending order. Both inputs are L2-normalised so cosine is a dot product. Useful: `np.argsort(...)`.

In [ ]:
def topk_retrieve(query_emb, all_embs, k):
    """Return indices of the k rows of all_embs most similar to query_emb."""
    sims = all_embs @ query_emb
    order = np.argsort(sims)[::-1]
    return order[:k]


_qq = np.array([1.0, 0.0])
_aa = np.array([[0.0, 1.0], [0.9, 0.1], [1.0, 0.0], [-1.0, 0.0]])
_aa = _aa / np.linalg.norm(_aa, axis=1, keepdims=True)
check_topk_retrieve(topk_retrieve, _qq, _aa, k=2, expected=np.array([2, 1]))

In [ ]:
# Compose the pieces: an attribute query none of the components was trained for.
if W_attr is None:
    print('⬜ Train W_attr above first.')
else:
    # Query: "round, with a closed loop, no straight strokes".
    # That is the canonical description of a 0 — but the contrastive bridge
    # was trained only on classes 2..9, so the digit 0 was never paired with
    # this attribute vector during training.
    query_attr = np.array([1, 0, 0, 0, 1], dtype=np.float64)
    print(f'Query: ' + '  '.join(f'{n}={int(v)}' for n, v in zip(attr_names, query_attr)))

    query_emb = query_attr @ W_attr
    query_emb = query_emb / (np.linalg.norm(query_emb) + 1e-8)

    # Embed every digit in the dataset, retrieve the top 60.
    _emb_all = encode(X_all, backbone)
    _top = topk_retrieve(query_emb, _emb_all, k=60)
    if _top is None:
        print('⬜ Implement topk_retrieve above first.')
    else:
        from collections import Counter
        _retrieved_labels = y_all[_top]
        _hist = Counter(_retrieved_labels.tolist())
        print(f'Retrieved label histogram (top 60): {dict(sorted(_hist.items()))}')

        # Cluster the retrieved embeddings to see if a coherent concept falls out.
        _km = KMeans(n_clusters=3, n_init=5, random_state=0).fit(_emb_all[_top])
        _cluster_sizes = Counter(_km.labels_.tolist())
        print(f'Cluster sizes : {dict(_cluster_sizes)}')
        for c in sorted(_cluster_sizes):
            _mask = _km.labels_ == c
            _members = Counter(_retrieved_labels[_mask].tolist())
            print(f'  cluster {c}: {dict(sorted(_members.items()))}')

        # Show the first 12 retrieved images.
        fig, axes = plt.subplots(2, 6, figsize=(10, 3.5))
        for ax, idx in zip(axes.ravel(), _top[:12]):
            ax.imshow(X_all[idx].reshape(8, 8), cmap='magma')
            ax.set_title(str(int(y_all[idx])), fontsize=9, color=_GOLDEN)
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values(): s.set_visible(False)
        plt.suptitle('Top retrievals for "round, has loop, no strokes"', y=1.02, color=_GOLDEN)
        plt.tight_layout()
        plt.show()

**Observe:**
- The top retrievals are dominated by 0s, even though no labelled 0 ever entered the training of any component used here. The attribute encoder learned its alignment on classes 2..9; k-means is the same algorithm from lecture 2; the backbone never had the "0" class.
- The composition produced a useful behaviour — class-free attribute search — that none of the components was individually built for.

**At scale: real foundation-model pipelines.** The same compositional pattern is what drives the most visible products built on foundation models today.

- **CLIP + Stable Diffusion**: CLIP's text encoder steers a generative diffusion model that was trained without any text supervision. The text → image prompt interface is glue, not training.
- **GroundingDINO + SAM**: an open-vocabulary detector finds bounding boxes for natural-language queries; SAM converts each box into a precise mask. Neither model was trained jointly with the other.
- **CLIP + an LLM**: image embedded by CLIP, top-K caption candidates retrieved from a corpus, an LLM reasons over them. Variants of this drive visual question answering and image-grounded chat.
- **An LLM + a code interpreter + a search tool**: the agentic frontier. The LLM is the reasoner, the tools are foundation-model-shaped pieces it was never explicitly trained alongside.

The pattern is always the same: pretrain large pieces on broad data, then *compose* them at inference time for tasks none of them was built for. That is the payoff of foundation models and it is the bridge to chapter 14, where the same recipe scales up to embodied action — Vision-Language-Action models that compose perception, language, and motor control.

### 🏁 Recap

**What we did:**
- 🧱 Pretrained a tiny MLP backbone on digits 2..9 and stripped the head to use the 16-d hidden activation as a general-purpose embedding. Walked through the historical line from hand-crafted features through ImageNet pretraining to web-scale foundation models.
- 🪜 Walked two points on the transfer continuum — frozen backbone with a linear probe, versus an MLP trained from scratch — and watched the probe dominate at low K. Named PEFT (LoRA, adapters, BitFit) as the same continuum at billion-parameter scale.
- 🎯 Reached unseen classes (0 and 1) through Lampert's attribute bridge, fitting a least-squares map from a hand-built attribute table to embedding centroids.
- 🔗 Replaced the hand-built bridge with a learned one via the diagonal contrastive loss — mini-CLIP — and saw the similarity matrix shift from noise to a lit diagonal. The recipe matches the published CLIP, SigLIP, and ALIGN models scaled six orders of magnitude smaller.
- 🧩 Composed the trained pieces into a free-form attribute query → retrieval → cluster pipeline that none of them was individually trained for, mirroring the way real systems compose CLIP + SAM + LLMs.

**Key takeaways:**
- A foundation model is a backbone with three additions — scale, breadth, and (often) cross-modal alignment. The architecture is not new; the data and the parameter count are.
- Transfer is a continuum parameterised by how much you trust the backbone and how much labelled target data you have. Frozen probe and full fine-tuning are the extremes; PEFT, prompting, and ICL fill in the middle for billion-parameter models.
- Zero-shot needs a bridge between class semantics and image features. Lampert built that bridge by hand from attributes; CLIP learns it from web-scale image-text pairs.
- Compositions of foundation models — CLIP + SAM, CLIP + Stable Diffusion, LLM + tools — are the dominant deployment pattern. None of the constituent models was trained for the composed task.

The next chapter, weak supervision, looks at what happens when you do have labels but they are noisy, partial, or generated by heuristics rather than experts. Chapter 14 picks up the composition thread again and scales it to embodied action.

## Take It from Here, Next Steps

These two extensions deepen the lecture beats without adding new infrastructure. Work through them at your own pace after the session.

### 🔬 Prompt ensembling analogue

Real CLIP gets a meaningful accuracy lift from averaging the embeddings of several phrasings of the same class — "a photo of a cat", "a picture of a cat", "an image of a cat". The attribute analogue: define two or three slightly different attribute rows for each unseen class (e.g. one with `vertical_stroke=1` for digit 1, one without — there is genuine ambiguity about whether a 1 has a serif), run zero-shot for each, and average the predicted centroids.

Try it: build `attrs_target_variants` as a dict mapping each class to a list of attribute rows, average the predicted centroids per class, and re-run zero-shot. The accuracy should move — sometimes up, sometimes down, depending on whether the variants point to the same place in embedding space.

### 🧪 Domain shift demo

The whole notebook lived inside a single distribution: 8×8 sklearn digits. Apply a simple corruption to the test set — a hue shift is meaningless on greyscale, but try Gaussian blur (`scipy.ndimage.gaussian_filter` with sigma 0.6) or salt-and-pepper noise. Re-run the K-shot transfer plot from section 🪜 with the corrupted test set, and the plot from section 🔗 (Lampert vs mini-CLIP).

The accuracies should drop everywhere, but the *gap* between strategies tells you something different: a frozen backbone is more robust to small input perturbations than a from-scratch model, because its features were learned on a richer source distribution. This is one face of *covariate shift* from the lecture's failure-mode beat — the input distribution moves while the label-given-input relationship stays fixed.

### 🦙 Run a real foundation model on your laptop

Everything above worked on a 5,000-parameter MLP backbone and an 80-parameter "text encoder". The mechanics are the published recipe, but they are easier to feel when the model on the other end is actually a real one. **Install [Ollama](https://ollama.com) and pull a small open-weights LLM** so that you have a foundation model running locally — no API key, no rate limit, no cloud round-trip. About 2–4 GB of disk and a few minutes are all it takes.

**Setup**

1. Download and install Ollama from [ollama.com](https://ollama.com) (Windows, macOS, and Linux installers).
2. Pull a small model that runs comfortably on a laptop CPU:
   - `ollama pull llama3.2:1b` (~1.3 GB, fastest, Meta)
   - `ollama pull phi3:mini` (~2.4 GB, strong reasoning for its size, Microsoft)
   - `ollama pull gemma2:2b` (~1.6 GB, well balanced, Google)
3. Sanity-check from a terminal: `ollama run llama3.2:1b "say hello in one word"`.

Once the Ollama daemon is running it exposes an HTTP API on `localhost:11434`, so you can drive it from Python without any extra dependency:

```python
import requests, json
resp = requests.post('http://localhost:11434/api/generate', json={
    'model': 'llama3.2:1b',
    'prompt': 'In one short sentence: what is a backbone in deep learning?',
    'stream': False,
})
print(resp.json()['response'])
```

**Exercise: in-context learning for digit attribution.** Section 🪜 named in-context learning as a point on the transfer continuum but did not let you touch it. Now you can.

1. **Zero-shot.** Ask the model: *"Describe the visual attributes of the digit 7 using these features: has_loop, two_loops, vertical_stroke, horizontal_stroke, curvy. Reply only with the values 0 or 1 separated by commas."* Compare the model's row to the `attrs_source[7]` we hard-coded in section 🎯.
2. **Few-shot via in-context examples.** Now provide three example rows in the prompt (digits 2, 5, 8 from `attrs_source`) before asking about a new digit. The model should latch onto the format and produce a cleaner answer. This is ICL: no gradient updates, behaviour shaped entirely by what you put in the prompt.
3. **Use the LLM as the bridge in section 🔗.** Replace the linear attribute encoder W with a call that asks the LLM "is this digit description consistent with the image features I have observed?" (you will need to encode the image features as a short text). The setup is hacky on purpose; the point is to feel that the LLM *is* a foundation model and slots into the same compositional pattern as CLIP did above.

**What you will notice.** Even a 1B-parameter model — three orders of magnitude smaller than the public hosted ones — produces coherent, structured answers. That is the foundation-model recipe paying off in your hands: a backbone trained on enough breadth that it generalises to a task it was never specifically built for, addressed entirely through the prompt. Same arc as the rest of this notebook, with one of the actual published artifacts on the other end of the wire.